# 📬 Email Campaign Analytics

A reusable workflow for the questions a marketing team actually asks after a send:

1. Which programs are getting clicks, and which are burning out the list (unsubscribes)?
2. Is performance trending up or down year over year?
3. Do English and French audiences respond differently?
4. Did the A/B test *really* win, or was it noise?

> **All data here is synthetic**, generated below with a fixed seed. No real customer or company data is used. The same code runs on a real export from Adobe Campaign or any ESP with `Campaignname`, `Sentdate`, `Delivered`, `Unique Clicks` and `Unsubscribe Clicks` columns.

In [ ]:
import re
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

## 1. Generate a realistic synthetic export
Campaign names follow a typical naming convention: `PROGRAM_LANG_VARIANT_YYYYMMDD`. Each program gets its own baseline click rate and unsubscribe rate, plus a little random noise, so the data behaves like the real thing.

In [ ]:
rng = np.random.default_rng(42)

# program -> (baseline CTR, baseline unsubscribe rate, typical audience size)
programs = {
    'WELCOME':    (0.062, 0.0020, 8_000),
    'NEWSLETTER': (0.021, 0.0035, 60_000),
    'PROMO':      (0.034, 0.0050, 45_000),
    'RETENTION':  (0.041, 0.0025, 20_000),
    'REENGAGE':   (0.012, 0.0080, 30_000),
}
year_trend = {2023: 1.00, 2024: 0.96, 2025: 1.05}   # small year-over-year drift
lang_mult  = {'EN': 1.00, 'FR': 1.08}              # French audience clicks a bit more
variant_mult = {'A': 1.00, 'B': 1.12}               # variant B is genuinely better

rows = []
for year in year_trend:
    for program, (ctr, unsub, size) in programs.items():
        for _ in range(24):  # ~2 sends a month per program
            date = pd.Timestamp(year, 1, 1) + pd.Timedelta(days=int(rng.integers(0, 365)))
            for lang in lang_mult:
                for variant in variant_mult:
                    delivered = int(rng.normal(size, size * 0.1) * (0.3 if lang == 'FR' else 0.7) / 2)
                    p_click = ctr * year_trend[year] * lang_mult[lang] * variant_mult[variant] * rng.normal(1, 0.08)
                    rows.append({
                        'Campaignname': f'{program}_{lang}_{variant}_{date:%Y%m%d}',
                        'Sentdate': date.strftime('%Y-%m-%d'),
                        'Delivered': delivered,
                        'Unique Clicks': int(rng.binomial(delivered, min(max(p_click, 0), 1))),
                        'Unsubscribe Clicks': int(rng.binomial(delivered, unsub)),
                    })

df = pd.DataFrame(rows)
print(f'{len(df):,} campaign sends generated')
df.head()

## 2. Clean and enrich
Real exports are messy (odd spacing, text in number columns), so clean defensively, then pull program, language and variant out of the campaign name.

In [ ]:
df.columns = df.columns.str.strip()
df['Campaignname'] = df['Campaignname'].astype(str).str.strip()
df['Sentdate'] = pd.to_datetime(df['Sentdate'], errors='coerce')
df['Year'] = df['Sentdate'].dt.year

for col in ['Delivered', 'Unique Clicks', 'Unsubscribe Clicks']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

def detect_language(name):
    name = name.upper()
    if re.search(r'(^|_)(FR|FRE|FRENCH)(_|$)', name):
        return 'French'
    if re.search(r'(^|_)(EN|ENG|ENGLISH)(_|$)', name):
        return 'English'
    return 'Other'

df['Program']  = df['Campaignname'].str.split('_').str[0]
df['Language'] = df['Campaignname'].apply(detect_language)
df['Variant']  = df['Campaignname'].str.split('_').str[2]
df.head()

## 3. Program scorecard
**CTR** = total unique clicks ÷ total delivered (a weighted rate, so one tiny send can't skew the average). **Unsub rate** is the list-health warning light.

In [ ]:
def summarize(data, by):
    s = data.groupby(by).agg(Sends=('Campaignname', 'count'),
                             Delivered=('Delivered', 'sum'),
                             Clicks=('Unique Clicks', 'sum'),
                             Unsubs=('Unsubscribe Clicks', 'sum')).reset_index()
    s['CTR %'] = (100 * s['Clicks'] / s['Delivered']).round(2)
    s['Unsub %'] = (100 * s['Unsubs'] / s['Delivered']).round(3)
    return s

scorecard = summarize(df, 'Program').sort_values('CTR %', ascending=False)
scorecard

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=scorecard, x='Unsub %', y='CTR %', size='Delivered', sizes=(100, 1200), legend=False, ax=ax)
for _, r in scorecard.iterrows():
    ax.annotate(r['Program'], (r['Unsub %'], r['CTR %']), xytext=(8, 4), textcoords='offset points')
ax.set_title('Engagement vs. list fatigue (bubble = volume)')
plt.show()

## 4. Year-over-year trend and language split

In [ ]:
yoy = summarize(df, ['Program', 'Year'])
lang = summarize(df, ['Language', 'Year'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=yoy, x='Program', y='CTR %', hue='Year', ax=axes[0])
axes[0].set_title('CTR by program and year')
sns.barplot(data=lang, x='Year', y='CTR %', hue='Language', ax=axes[1])
axes[1].set_title('English vs. French CTR')
plt.tight_layout()
plt.show()

## 5. Did variant B really win?
A bigger CTR isn't proof. A **two-proportion z-test** asks: if A and B were truly equal, how often would we see a gap this big just by luck? That chance is the p-value; below 0.05 is the usual bar.

In [ ]:
def ab_test(clicks_a, sent_a, clicks_b, sent_b):
    p_a, p_b = clicks_a / sent_a, clicks_b / sent_b
    pooled = (clicks_a + clicks_b) / (sent_a + sent_b)
    se = math.sqrt(pooled * (1 - pooled) * (1 / sent_a + 1 / sent_b))
    z = (p_b - p_a) / se
    p_value = math.erfc(abs(z) / math.sqrt(2))  # two-sided
    return p_a, p_b, (p_b / p_a - 1), p_value

results = []
for program, g in df.groupby('Program'):
    a = g[g['Variant'] == 'A'][['Unique Clicks', 'Delivered']].sum()
    b = g[g['Variant'] == 'B'][['Unique Clicks', 'Delivered']].sum()
    p_a, p_b, lift, p = ab_test(a['Unique Clicks'], a['Delivered'], b['Unique Clicks'], b['Delivered'])
    results.append({'Program': program, 'CTR A %': round(100 * p_a, 2), 'CTR B %': round(100 * p_b, 2),
                    'Lift': f'{lift:+.1%}', 'p-value': f'{p:.2g}', 'Verdict': 'B wins' if p < 0.05 and lift > 0 else 'Not proven'})

pd.DataFrame(results)

## 6. What I'd tell the marketing team
- **Welcome and Retention** earn the best clicks with low unsubscribes, so protect and expand them.
- **Re-engagement** has the worst click-to-unsubscribe trade-off. Tighten who gets it (for example, only people inactive 90–180 days) instead of blasting the whole dormant list.
- **French audiences click more.** Worth checking whether French content gets the same testing and attention as English.
- **Variant B is a real winner** across programs, so roll it out, and keep testing the next idea against it.

*(These conclusions come from the synthetic data above; on real data, the same notebook produces the real answers.)*